In [ ]:
import os

# Fayllar to'liq yuklanganini tekshiramiz
video_name = "Replay 2026-07-07 21-42-56.mp4"
video_name2 = "Replay 2026-07-07 21-31-03.mp4"
video_name3 = "penalty.mp4"
video_name4 = "por_isp.mp4"
video_name5 = "Uzb_Kon.mp4"
video_name6 = "Uzb_Col.mp4"
video_name7 = "Uzb_Por.mp4"
video_name8 = "Eng_Kon.mp4"
video_name = "download.mp4"
model_name = "footballdetm2.pt"

assert os.path.exists(video_name8), f"Video topilmadi: {video_name}"
assert os.path.exists(model_name), f"Model topilmadi: {model_name}"
print("Ikkala fayl ham joyida.")

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO

model = YOLO(model_name)
print("Model sinflari:", model.names)

results = model.track(
    source=video_name,
    tracker="bytetrack.yaml",
    conf=0.3,
    save=True
)

print("Natija saqlangan papka:", results[0].save_dir)

In [ ]:
import glob

output_video = glob.glob(f"{results[0].save_dir}/*.mp4") + glob.glob(f"{results[0].save_dir}/*.avi")
print("Topilgan natija fayli:", output_video)

# Colab ba'zan .avi formatda saqlaydi, uni .mp4 ga o'giramiz (brauzerda ko'rinishi uchun)
if output_video and output_video[0].endswith(".avi"):
    converted_path = output_video[0].replace(".avi", "_converted.mp4")
    os.system(f'ffmpeg -y -i "{output_video[0]}" -vcodec libx264 "{converted_path}"')
    output_video = [converted_path]

from IPython.display import Video
Video(output_video[0], embed=True, width=800)

# **Model dizayniga biroz o'zgartirish kiritildi**

In [ ]:
!pip install ultralytics opencv-python -q

In [ ]:
!pip install yt-dlp -q

In [ ]:
!yt-dlp -f "best[ext=mp4]" -o "test_video.mp4" "https://youtube.com/shorts/7GBz082psxQ"

In [ ]:
"""
custom_football_viz.py
------------------------
Ultralytics track() natijalarini olib, o'zimizning maxsus
vizualizatsiyamizni chizadi:

1) To'rtburchak (bbox) o'rniga - oyoq ostida ELLIPS (to'p bundan mustasno)
2) Yorliq qisqartirilgan: "12-P-87%" (id - Rol harfi - ishonch foizi)
   Rol harflari: P=player, G=goalkeeper, R=referee, B=ball, O=other
3) To'p ortidan RANGLI IZ (trail) - qaysi jamoa egalik qilsa o'sha
   rangda, va bir necha soniyadan keyin AVTOMATIK O'CHIB KETADI
4) Jamoa (qizil/ko'k) - kiyim rangiga qarab K-means orqali AVTOMATIK
   aniqlanadi (alohida annotatsiya kerak emas)

ISHLATISH (Colab):
    python custom_football_viz.py
    (yoki katakchaga shu faylning mazmunini joylab ishga tushiring)
"""

import cv2
import numpy as np
from collections import deque
from ultralytics import YOLO

# ---------------------------------------------------------
# SOZLAMALAR
# ---------------------------------------------------------
VIDEO_PATH = "Replay 2026-07-07 21-42-56.mp4"      # kiruvchi video
# VIDEO_PATH = "test_video.mp4"
MODEL_PATH = "footballdetm2.pt"                    # sizning modelingiz
OUTPUT_PATH = "output_custom.mp4"

CONF_THRESHOLD = 0.3
BALL_TRAIL_SECONDS = 2.5      # to'p izi necha soniyadan keyin o'chadi
POSSESSION_DIST_THRESHOLD = 80  # piksel - to'p shu masofadan yaqin o'yinchi "egasi"

# Rol -> qisqa harf
ROLE_LETTER = {
    "player": "P",
    "goalkeeper": "G",
    "referee": "R",
    "ball": "B",
    "other": "O",
}

TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]  # qizil, ko'k (BGR formatda)


# ---------------------------------------------------------
# YORDAMCHI FUNKSIYALAR
# ---------------------------------------------------------
def get_shirt_color(frame, box):
    """Bounding box'ning yuqori 40% qismidan (ko'ylak) o'rtacha rangni oladi."""
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = y1 + int(h * 0.4)
    shirt_y2 = max(shirt_y2, y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    """Oyoq ostiga ellips chizadi (to'rtburchak o'rniga)."""
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(
        frame, center, (width // 2, int(width * 0.2)), 0, -45, 235,
        color, thickness, cv2.LINE_4,
    )
    return frame


def draw_label(frame, box, text, color):
    """Qisqartirilgan yorliqni chizadi: masalan '12-P-87%'."""
    x1, y1, x2, y2 = map(int, box)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = y1 - th - 8
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), color, -1)
    cv2.putText(
        frame, text, (cx - tw // 2, top + th + 1),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA,
    )
    return frame


# ---------------------------------------------------------
# ASOSIY JARAYON
# ---------------------------------------------------------
def main():
    model = YOLO(MODEL_PATH)
    names = model.names  # {0:'player', 1:'goalkeeper', ...} - sizning sinflaringiz

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    trail_max_len = int(fps * BALL_TRAIL_SECONDS)
    ball_trail = deque(maxlen=trail_max_len)  # har biri: (x, y, team_color)

    # K-means jamoa klasteri - dastlab None, birinchi bir necha kadrda o'rganiladi
    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(
        source=VIDEO_PATH,
        tracker="bytetrack.yaml",
        conf=CONF_THRESHOLD,
        stream=True,   # xotirani tejash uchun kadr-kadr qayta ishlaymiz
    )

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        # --- 1-bosqich: dastlabki 30 kadrda jamoa ranglarini yig'amiz ---
        if team_centers is None and boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                if role == "player":
                    xyxy = box.xyxy[0].cpu().numpy()
                    color_samples.append(get_shirt_color(frame, xyxy))

            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, labels, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers  # 2 ta o'rtacha rang (jamoa A, jamoa B)

        ball_center = None
        player_positions = []  # (x, y, team_idx)

        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = xyxy

                if role == "ball":
                    # To'p uchun oddiy doira (ellips emas) chizamiz
                    bx, by = int((x1 + x2) / 2), int((y1 + y2) / 2)
                    ball_center = (bx, by)
                    cv2.circle(frame, (bx, by), 6, (0, 255, 255), -1)
                    label = f"{track_id}-{ROLE_LETTER['ball']}-{int(conf*100)}%"
                    draw_label(frame, xyxy, label, (0, 255, 255))
                    continue

                # Jamoa rangini aniqlaymiz (agar player bo'lsa va klasterlar tayyor bo'lsa)
                team_idx = 0
                if role == "player" and team_centers is not None:
                    shirt_color = get_shirt_color(frame, xyxy)
                    dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                    team_idx = int(np.argmin(dists))
                    player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

                color = TEAM_COLORS[team_idx] if role == "player" else (255, 255, 255)
                draw_ellipse(frame, xyxy, color)

                label = f"{track_id}-{ROLE_LETTER.get(role, 'O')}-{int(conf*100)}%"
                draw_label(frame, xyxy, label, color)

        # --- To'p egasini aniqlab, iz ranggini belgilaymiz ---
        if ball_center is not None:
            possessor_team = None
            min_dist = POSSESSION_DIST_THRESHOLD
            for (px, py, t_idx) in player_positions:
                d = np.hypot(ball_center[0] - px, ball_center[1] - py)
                if d < min_dist:
                    min_dist = d
                    possessor_team = t_idx

            trail_color = TEAM_COLORS[possessor_team] if possessor_team is not None else (200, 200, 200)
            ball_trail.append((ball_center[0], ball_center[1], trail_color))

        # --- To'p izini chizamiz (asta-sekin xiralashadi) ---
        for i in range(1, len(ball_trail)):
            x1p, y1p, c1 = ball_trail[i - 1]
            x2p, y2p, _ = ball_trail[i]
            alpha = i / len(ball_trail)  # eskisi xira, yangisi yorqin
            thickness = max(1, int(4 * alpha))
            overlay = frame.copy()
            cv2.line(overlay, (int(x1p), int(y1p)), (int(x2p), int(y2p)), c1, thickness)
            cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom.mp4 -vcodec libx264 output_custom_converted.mp4')
Video("output_custom_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
TUZATILDI:
1) Har bir rol uchun ALOHIDA, aniq ko'rinadigan rang (fonga qorishmaydi)
2) To'p izi uchun IKKI QATLAM HIMOYA:
   - Past ishonchdagi (confidence) noto'g'ri aniqlanishlar filtrlanadi
   - Agar to'p mantiqsiz uzoqqa "sakrasa" (teleport) - rad etiladi
3) Yorliqlar endi qorong'i qalin fon bilan, har doim o'qiladigan
"""

import cv2
import numpy as np
from collections import deque
from ultralytics import YOLO

# ---------------------------------------------------------
# SOZLAMALAR
# ---------------------------------------------------------
VIDEO_PATH = "Replay 2026-07-07 21-42-56.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom.mp4"

CONF_THRESHOLD = 0.3          # umumiy aniqlash chegarasi
BALL_CONF_THRESHOLD = 0.45    # to'p uchun QATTIQROQ chegara - soxta aniqlanishni kamaytiradi
BALL_TRAIL_SECONDS = 2.5
POSSESSION_DIST_THRESHOLD = 80
MAX_BALL_JUMP_PX = 120        # to'p bir kadrda shu masofadan ko'p "sakrasa" - rad etiladi

ROLE_LETTER = {
    "player": "P",
    "goalkeeper": "G",
    "referee": "R",
    "ball": "B",
    "other": "O",
}

# Har bir rol uchun ALOHIDA, kontrastli rang (BGR format)
# Fonga (yashil maydon) qorishmaydigan, bir-biridan farqlanadigan ranglar tanlandi
ROLE_COLORS = {
    "goalkeeper": (0, 255, 255),   # och sariq
    "referee":    (0, 0, 0),        # qora (hakam formasi odatda qora/och rang bo'ladi, shuning uchun ajralib turadi)
    "other":      (180, 180, 180),  # kulrang
}
REFEREE_BOX_BG = (0, 255, 0)  # hakam yorlig'i uchun fon rangi - yorqin yashil, qora matn bilan yaxshi ko'rinadi

TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]  # qizil, ko'k


# ---------------------------------------------------------
# YORDAMCHI FUNKSIYALAR
# ---------------------------------------------------------
def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = y1 + int(h * 0.4)
    shirt_y2 = max(shirt_y2, y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(
        frame, center, (width // 2, int(width * 0.2)), 0, -45, 235,
        color, thickness, cv2.LINE_4,
    )
    return frame


def draw_label(frame, box, text, bg_color, text_color=(255, 255, 255)):
    """
    TUZATILDI: matn rangi endi fon ranggiga qarab avtomatik tanlanadi -
    och fon bo'lsa qora matn, quyuq fon bo'lsa oq matn - har doim o'qiladi.
    """
    x1, y1, x2, y2 = map(int, box)

    # Fon yorqinligiga qarab matn rangini avtomatik tanlaymiz (o'qilishi uchun)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)

    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = y1 - th - 8
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(
        frame, text, (cx - tw // 2, top + th + 1),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA,
    )
    return frame


# ---------------------------------------------------------
# ASOSIY JARAYON
# ---------------------------------------------------------
def main():
    model = YOLO(MODEL_PATH)
    names = model.names

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    trail_max_len = int(fps * BALL_TRAIL_SECONDS)
    ball_trail = deque(maxlen=trail_max_len)
    last_ball_pos = None   # TUZATILDI: oxirgi haqiqiy to'p pozitsiyasini kuzatib boramiz

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(
        source=VIDEO_PATH,
        tracker="bytetrack.yaml",
        conf=CONF_THRESHOLD,
        stream=True,
    )

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        if team_centers is None and boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                if role == "player":
                    xyxy = box.xyxy[0].cpu().numpy()
                    color_samples.append(get_shirt_color(frame, xyxy))

            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, labels, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        ball_center = None
        best_ball_conf = 0.0
        best_ball_box = None
        player_positions = []

        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = xyxy

                if role == "ball":
                    # TUZATILDI: kadrda BITTA to'p bo'lishi kerak - eng ishonchlisini tanlaymiz,
                    # va past ishonchdagilarni butunlay e'tiborsiz qoldiramiz
                    if conf >= BALL_CONF_THRESHOLD and conf > best_ball_conf:
                        best_ball_conf = conf
                        best_ball_box = (x1, y1, x2, y2, track_id, conf)
                    continue

                team_idx = 0
                if role == "player" and team_centers is not None:
                    shirt_color = get_shirt_color(frame, xyxy)
                    dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                    team_idx = int(np.argmin(dists))
                    player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

                # TUZATILDI: har bir rol uchun aniq, kontrastli rang
                if role == "player":
                    color = TEAM_COLORS[team_idx]
                elif role == "referee":
                    color = REFEREE_BOX_BG
                else:
                    color = ROLE_COLORS.get(role, (180, 180, 180))

                draw_ellipse(frame, xyxy, color)
                label = f"{track_id}-{ROLE_LETTER.get(role, 'O')}-{int(conf*100)}%"
                draw_label(frame, xyxy, label, color)

        # --- To'pni qayta ishlash: soxta aniqlanish va "sakrash"ni filtrlaymiz ---
        if best_ball_box is not None:
            x1, y1, x2, y2, track_id, conf = best_ball_box
            bx, by = int((x1 + x2) / 2), int((y1 + y2) / 2)

            # TUZATILDI: agar oldingi pozitsiyadan mantiqsiz uzoqqa "sakragan" bo'lsa,
            # bu ehtimol soxta aniqlanish (masalan forma logotipi) - rad etamiz
            is_valid = True
            if last_ball_pos is not None:
                jump_dist = np.hypot(bx - last_ball_pos[0], by - last_ball_pos[1])
                if jump_dist > MAX_BALL_JUMP_PX:
                    is_valid = False

            if is_valid:
                ball_center = (bx, by)
                last_ball_pos = (bx, by)
                cv2.circle(frame, (bx, by), 6, (0, 255, 255), -1)
                label = f"{track_id}-{ROLE_LETTER['ball']}-{int(conf*100)}%"
                draw_label(frame, (x1, y1, x2, y2), label, (0, 200, 255))

        if ball_center is not None:
            possessor_team = None
            min_dist = POSSESSION_DIST_THRESHOLD
            for (px, py, t_idx) in player_positions:
                d = np.hypot(ball_center[0] - px, ball_center[1] - py)
                if d < min_dist:
                    min_dist = d
                    possessor_team = t_idx

            trail_color = TEAM_COLORS[possessor_team] if possessor_team is not None else (200, 200, 200)
            ball_trail.append((ball_center[0], ball_center[1], trail_color))

        for i in range(1, len(ball_trail)):
            x1p, y1p, c1 = ball_trail[i - 1]
            x2p, y2p, _ = ball_trail[i]
            alpha = i / len(ball_trail)
            thickness = max(1, int(4 * alpha))
            overlay = frame.copy()
            cv2.line(overlay, (int(x1p), int(y1p)), (int(x2p), int(y2p)), c1, thickness)
            cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom.mp4 -vcodec libx264 output_custom_converted.mp4')
Video("output_custom_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI TUZATISH:
1) To'p uchun endi TO'RTBURCHAK (bounding box) chiziladi, doira o'rniga
2) To'p yorlig'i har doim aniq ko'rinadi (fon rangi bilan)
3) Ishonch chegarasi va "sakrash" cheklovi YUMSHATILDI - to'p yo'qolib qolmaydi
4) Agar to'p bir necha kadr ko'rinmasa, keyingi ko'rinishda "sakrash" tekshiruvi
   qayta boshlanadi (eski pozitsiya unutiladi)
5) Iz (trail) TEZROQ o'chadigan qilindi
"""

import cv2
import numpy as np
from collections import deque
from ultralytics import YOLO

# ---------------------------------------------------------
# SOZLAMALAR
# ---------------------------------------------------------
VIDEO_PATH = "Replay 2026-07-07 21-42-56.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_3.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25    # TUZATILDI: pasaytirildi (0.45 -> 0.25), to'p ko'proq topiladi
BALL_TRAIL_SECONDS = 0.8      # TUZATILDI: qisqartirildi (2.5 -> 0.8), tezroq o'chadi
POSSESSION_DIST_THRESHOLD = 80
MAX_BALL_JUMP_PX = 250        # TUZATILDI: kattalashtirildi (120 -> 250), tez uchgan to'pni rad etmaydi
MAX_MISSED_FRAMES = 5         # YANGI: shuncha kadr to'p ko'rinmasa, eski pozitsiya unutiladi

ROLE_LETTER = {
    "player": "P",
    "goalkeeper": "G",
    "referee": "R",
    "ball": "B",
    "other": "O",
}

ROLE_COLORS = {
    "goalkeeper": (0, 255, 255),
    "referee":    (0, 255, 0),
    "other":      (180, 180, 180),
}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)     # to'q sariq/apelsin - to'p uchun ajralib turadigan rang

TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


# ---------------------------------------------------------
# YORDAMCHI FUNKSIYALAR
# ---------------------------------------------------------
def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = y1 + int(h * 0.4)
    shirt_y2 = max(shirt_y2, y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(
        frame, center, (width // 2, int(width * 0.2)), 0, -45, 235,
        color, thickness, cv2.LINE_4,
    )
    return frame


def draw_box(frame, box, color, thickness=2):
    """YANGI: to'p uchun to'rtburchak (rectangle) chizadi."""
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)

    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = y1 - th - 8
    top = max(top, 0)  # ekrandan tashqariga chiqmasin
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(
        frame, text, (cx - tw // 2, top + th + 1),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA,
    )
    return frame


# ---------------------------------------------------------
# ASOSIY JARAYON
# ---------------------------------------------------------
def main():
    model = YOLO(MODEL_PATH)
    names = model.names

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    trail_max_len = max(2, int(fps * BALL_TRAIL_SECONDS))
    ball_trail = deque(maxlen=trail_max_len)
    last_ball_pos = None
    missed_frames = 0   # YANGI: to'p necha kadrdir ko'rinmaganini sanaymiz

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(
        source=VIDEO_PATH,
        tracker="bytetrack.yaml",
        conf=CONF_THRESHOLD,
        stream=True,
    )

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        if team_centers is None and boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                if role == "player":
                    xyxy = box.xyxy[0].cpu().numpy()
                    color_samples.append(get_shirt_color(frame, xyxy))

            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, labels, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        ball_center = None
        best_ball_conf = 0.0
        best_ball_box = None
        player_positions = []

        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = xyxy

                if role == "ball":
                    if conf >= BALL_CONF_THRESHOLD and conf > best_ball_conf:
                        best_ball_conf = conf
                        best_ball_box = (x1, y1, x2, y2, track_id, conf)
                    continue

                team_idx = 0
                if role == "player" and team_centers is not None:
                    shirt_color = get_shirt_color(frame, xyxy)
                    dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                    team_idx = int(np.argmin(dists))
                    player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

                if role == "player":
                    color = TEAM_COLORS[team_idx]
                elif role == "referee":
                    color = REFEREE_BOX_BG
                else:
                    color = ROLE_COLORS.get(role, (180, 180, 180))

                draw_ellipse(frame, xyxy, color)
                label = f"{track_id}-{ROLE_LETTER.get(role, 'O')}-{int(conf*100)}%"
                draw_label(frame, xyxy, label, color)

        # --- To'pni qayta ishlash ---
        if best_ball_box is not None:
            x1, y1, x2, y2, track_id, conf = best_ball_box
            bx, by = int((x1 + x2) / 2), int((y1 + y2) / 2)

            is_valid = True
            # TUZATILDI: agar to'p uzoq vaqt ko'rinmagan bo'lsa (missed_frames katta),
            # "sakrash" tekshiruvini o'tkazib yuboramiz - bu YANGI, mustaqil aniqlanish
            if last_ball_pos is not None and missed_frames < MAX_MISSED_FRAMES:
                jump_dist = np.hypot(bx - last_ball_pos[0], by - last_ball_pos[1])
                if jump_dist > MAX_BALL_JUMP_PX:
                    is_valid = False

            if is_valid:
                ball_center = (bx, by)
                last_ball_pos = (bx, by)
                missed_frames = 0

                # YANGI: to'rtburchak chizamiz (doira o'rniga)
                draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, thickness=2)
                label = f"{track_id}-{ROLE_LETTER['ball']}-{int(conf*100)}%"
                draw_label(frame, (x1, y1, x2, y2), label, BALL_COLOR)
            else:
                missed_frames += 1
        else:
            missed_frames += 1

        if ball_center is not None:
            possessor_team = None
            min_dist = POSSESSION_DIST_THRESHOLD
            for (px, py, t_idx) in player_positions:
                d = np.hypot(ball_center[0] - px, ball_center[1] - py)
                if d < min_dist:
                    min_dist = d
                    possessor_team = t_idx

            trail_color = TEAM_COLORS[possessor_team] if possessor_team is not None else (200, 200, 200)
            ball_trail.append((ball_center[0], ball_center[1], trail_color))

        for i in range(1, len(ball_trail)):
            x1p, y1p, c1 = ball_trail[i - 1]
            x2p, y2p, _ = ball_trail[i]
            alpha = i / len(ball_trail)
            thickness = max(1, int(3 * alpha))
            overlay = frame.copy()
            cv2.line(overlay, (int(x1p), int(y1p)), (int(x2p), int(y2p)), c1, thickness)
            cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_3.mp4 -vcodec libx264 output_custom_3_converted.mp4')
Video("output_custom_3_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI: Kalman filter qo'shildi - to'p izini SILLIQ va BARQAROR qiladi.
Bu - professional sport-tracking tizimlarida ishlatiladigan standart usul.
"""

import cv2
import numpy as np
from collections import deque
from ultralytics import YOLO

# ---------------------------------------------------------
# SOZLAMALAR
# ---------------------------------------------------------
VIDEO_PATH = "Replay 2026-07-07 21-42-56.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_4.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
BALL_TRAIL_SECONDS = 0.8
POSSESSION_DIST_THRESHOLD = 80

SHOW_BALL_TRAIL = False  # False qilsangiz - iz umuman chizilmaydi, faqat to'rtburchak qoladi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


# ---------------------------------------------------------
# KALMAN FILTER - to'p harakatini silliqlash uchun
# ---------------------------------------------------------
class BallKalmanTracker:
    """
    To'pning (x, y) pozitsiyasi va tezligini kuzatib boradi.
    Har bir yangi o'lchov (detection) kelganda, buni o'zining
    bashorati bilan solishtirib, natijani "silliqlashtiradi".
    Agar o'lchov bir necha kadr kelmasa, faqat bashorat asosida davom etadi.
    """
    def __init__(self):
        self.kf = cv2.KalmanFilter(4, 2)  # holat: [x, y, vx, vy], o'lchov: [x, y]
        self.kf.measurementMatrix = np.array([[1, 0, 0, 0],
                                                [0, 1, 0, 0]], np.float32)
        self.kf.transitionMatrix = np.array([[1, 0, 1, 0],
                                              [0, 1, 0, 1],
                                              [0, 0, 1, 0],
                                              [0, 0, 0, 1]], np.float32)
        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1.5
        self.initialized = False
        self.missed_frames = 0
        self.max_missed = 15  # shuncha kadrdan keyin trekni "yo'qotilgan" deb hisoblaymiz

    def update(self, measurement):
        """measurement=(x,y) yoki None (agar shu kadrda to'p topilmagan bo'lsa)"""
        predicted = self.kf.predict()
        pred_x, pred_y = float(predicted[0]), float(predicted[1])

        if measurement is not None:
            mx, my = measurement
            if not self.initialized:
                self.kf.statePre = np.array([[mx], [my], [0], [0]], np.float32)
                self.kf.statePost = np.array([[mx], [my], [0], [0]], np.float32)
                self.initialized = True
                self.missed_frames = 0
                return (mx, my)

            # TUZATISH: agar o'lchov bashoratdan JUDA uzoq bo'lsa, bu soxta aniqlanish -
            # o'lchovni rad etamiz, faqat bashoratga ishonamiz
            dist = np.hypot(mx - pred_x, my - pred_y)
            if dist > 200 and self.missed_frames < 3:
                self.missed_frames += 1
                return (pred_x, pred_y) if self.missed_frames <= self.max_missed else None

            corrected = self.kf.correct(np.array([[np.float32(mx)], [np.float32(my)]]))
            self.missed_frames = 0
            return (float(corrected[0]), float(corrected[1]))
        else:
            # Bu kadrda to'p topilmadi - faqat bashorat bilan davom etamiz
            self.missed_frames += 1
            if not self.initialized or self.missed_frames > self.max_missed:
                return None
            return (pred_x, pred_y)


# ---------------------------------------------------------
# YORDAMCHI FUNKSIYALAR (o'zgarishsiz)
# ---------------------------------------------------------
def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


# ---------------------------------------------------------
# ASOSIY JARAYON
# ---------------------------------------------------------
def main():
    model = YOLO(MODEL_PATH)
    names = model.names

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    trail_max_len = max(2, int(fps * BALL_TRAIL_SECONDS))
    ball_trail = deque(maxlen=trail_max_len)
    ball_kalman = BallKalmanTracker()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        if team_centers is None and boxes is not None:
            for box in boxes:
                if names[int(box.cls[0])] == "player":
                    color_samples.append(get_shirt_color(frame, box.xyxy[0].cpu().numpy()))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball_conf = 0.0
        best_ball_box = None
        player_positions = []

        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                role = names[cls_id]
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = xyxy

                if role == "ball":
                    if conf >= BALL_CONF_THRESHOLD and conf > best_ball_conf:
                        best_ball_conf = conf
                        best_ball_box = (x1, y1, x2, y2, track_id, conf)
                    continue

                team_idx = 0
                if role == "player" and team_centers is not None:
                    shirt_color = get_shirt_color(frame, xyxy)
                    dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                    team_idx = int(np.argmin(dists))
                    player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

                color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
                draw_ellipse(frame, xyxy, color)
                draw_label(frame, xyxy, f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        # --- Kalman filter orqali to'p pozitsiyasini silliqlaymiz ---
        raw_measurement = None
        raw_box_for_draw = None
        if best_ball_box is not None:
            x1, y1, x2, y2, track_id, conf = best_ball_box
            raw_measurement = ((x1 + x2) / 2, (y1 + y2) / 2)
            raw_box_for_draw = (x1, y1, x2, y2, track_id, conf)

        smoothed_pos = ball_kalman.update(raw_measurement)

        if smoothed_pos is not None:
            sx, sy = smoothed_pos
            if raw_box_for_draw is not None:
                # Haqiqiy aniqlanish bo'lsa - asl to'rtburchakni chizamiz
                x1, y1, x2, y2, track_id, conf = raw_box_for_draw
                draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
                draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)
            else:
                # Faqat bashorat bo'lsa - kichikroq, xiraroq doira (ko'rinmagan payt uchun)
                cv2.circle(frame, (int(sx), int(sy)), 8, BALL_COLOR, 2)

            if SHOW_BALL_TRAIL:
                possessor_team = None
                min_dist = POSSESSION_DIST_THRESHOLD
                for (px, py, t_idx) in player_positions:
                    d = np.hypot(sx - px, sy - py)
                    if d < min_dist:
                        min_dist = d
                        possessor_team = t_idx
                trail_color = TEAM_COLORS[possessor_team] if possessor_team is not None else (200, 200, 200)
                ball_trail.append((sx, sy, trail_color))

        if SHOW_BALL_TRAIL:
            for i in range(1, len(ball_trail)):
                x1p, y1p, c1 = ball_trail[i - 1]
                x2p, y2p, _ = ball_trail[i]
                alpha = i / len(ball_trail)
                thickness = max(1, int(3 * alpha))
                overlay = frame.copy()
                cv2.line(overlay, (int(x1p), int(y1p)), (int(x2p), int(y2p)), c1, thickness)
                cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_4.mp4 -vcodec libx264 output_custom_4_converted.mp4')
Video("output_custom_4_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "Replay 2026-07-07 21-42-56.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_5.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_5.mp4 -vcodec libx264 output_custom_5_converted.mp4')
Video("output_custom_5_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "penalty.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_6.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_6.mp4 -vcodec libx264 output_custom_6_converted.mp4')
Video("output_custom_6_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "por_isp.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_7.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_7.mp4 -vcodec libx264 output_custom_7_converted.mp4')
Video("output_custom_7_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "Uzb_Col.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_8.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_8.mp4 -vcodec libx264 output_custom_8_converted.mp4')
Video("output_custom_8_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "Uzb_Kon.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_9.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_9.mp4 -vcodec libx264 output_custom_9_converted.mp4')
Video("output_custom_9_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "Uzb_Por.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_10.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_10.mp4 -vcodec libx264 output_custom_10_converted.mp4')
Video("output_custom_10_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py
------------------------
YANGI:
1) Dublikat box'lar (bir odamga 2 marta chizish) - IKKALASI HAM olib tashlanadi
2) Rol chalkashishi (darvozabon/hakam) - VAQTINCHALIK SILLIQLASH orqali kamaytirildi:
   har bir track_id uchun oxirgi 15 kadrdagi rol tarixini saqlab,
   "ko'pchilik ovozi" bilan qaror qabul qilinadi
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

VIDEO_PATH = "Eng_Kon.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_13.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15   # necha kadr tarixi asosida rol qarori qabul qilinadi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
ROLE_COLORS = {"goalkeeper": (0, 255, 255), "referee": (0, 255, 0), "other": (180, 180, 180)}
REFEREE_BOX_BG = (0, 255, 0)
BALL_COLOR = (0, 165, 255)
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]


def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi."""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            cls_i = boxes_list[i][0]
            cls_j = boxes_list[j][0]
            if cls_i != cls_j:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


class RoleSmoother:
    """
    Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi. Bu - bitta kadrdagi
    tasodifiy xato (masalan darvozabon->hakam) ni silliqlaydi.
    """
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_box(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 150 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True)

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        best_ball = None
        player_positions = []

        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]

            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue

            # --- YANGI: rolni vaqt bo'yicha silliqlaymiz ---
            role = role_smoother.update_and_get(track_id, raw_role)

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, (x1, y1, x2, y2))
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))
                player_positions.append(((x1 + x2) / 2, (y1 + y2) / 2, team_idx))

            color = TEAM_COLORS[team_idx] if role == "player" else (REFEREE_BOX_BG if role == "referee" else ROLE_COLORS.get(role, (180, 180, 180)))
            draw_ellipse(frame, (x1, y1, x2, y2), color)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-{ROLE_LETTER.get(role,'O')}-{int(conf*100)}%", color)

        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            draw_box(frame, (x1, y1, x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{track_id}-B-{int(conf*100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_13.mp4 -vcodec libx264 output_custom_13_converted.mp4')
Video("output_custom_13_converted.mp4", embed=True, width=800)

In [ ]:
"""
custom_football_viz.py  (v4 - YAKUNIY BIRLASHTIRILGAN VERSIYA)
--------------------------------------------------------------------------
Bu versiyada 3 XIL TUZATISH MEXANIZMI birlashtirilgan, har biri O'ZINING
XATO TURINI hal qiladi:

  1) DUBLIKAT BOX FILTRI (filter_overlapping_boxes)
     Muammo: bitta odamga model ba'zan 2 marta box chizadi (ustma-ust).
     Yechim: bir xil klassdagi, bir-biriga juda yaqin (IOU yuqori) ikkita
     box topilsa - IKKALASI HAM olib tashlanadi (qaysi biri to'g'ri
     ekanini aniq bilmaganimiz uchun, ikkalasini saqlab, keyingi bosqichda
     "goalkeeper zonasi" yoki "rol silliqlash" orqali xato tuzatilishidan
     ko'ra, boshidanoq ikkalasini olib tashlash xavfsizroq).

  2) GOALKEEPER ZONA QOIDASI (ISHONCH FOIZIDAN QAT'I NAZAR ishlaydi)
     Muammo: model ba'zan markaziy maydondagi ODDIY o'yinchini "goalkeeper"
     deb YUQORI ishonch bilan DOIMIY xato qiladi (bu bir martalik
     tasodifiy xato emas - ko'p kadrlar davomida barqaror xato, shuning
     uchun "rol silliqlash" buni tuzata OLMAYDI).
     Yechim: darvozabon deyarli har doim darvoza zonasida turadi - bu
     futbolning o'zgarmas jismoniy qoidasi. Agar "goalkeeper" deb
     topilgan odam darvoza zonasidan uzoqda bo'lsa - ishonch foizidan
     qat'i nazar, "player"ga qaytariladi.

  3) ROL SILLIQLASH (RoleSmoother, 15 kadr tarixi)
     Muammo: bitta-ikkita kadrda tasodifiy "chayqalish" (masalan, bir
     lahzaga hakam->o'yinchi bo'lib ko'rinishi).
     Yechim: har bir track_id uchun oxirgi 15 kadrdagi rol tarixini
     saqlab, "ko'pchilik ovozi" bilan barqaror rolni tanlaydi.

ISHLASH TARTIBI (har bir kadr uchun):
    xom aniqlashlar -> dublikat filtri -> zona qoidasi + rang heuristikasi
    (past ishonchda) -> rol silliqlash (vaqt bo'yicha) -> chizish

Bundan tashqari, TO'P ko'rsatilishi soddalashtirilgan:
    - Iz (trail) chizilmaydi
    - To'rtburchak (bbox) chiziladi
    - Yorliqda ID ko'rsatilmaydi, faqat "B-87%" kabi

ISHLATISH (Colab):
    python custom_football_viz.py
"""

import cv2
import numpy as np
from collections import deque, defaultdict, Counter
from ultralytics import YOLO

# ---------------------------------------------------------
# SOZLAMALAR
# ---------------------------------------------------------
VIDEO_PATH = "Eng_Kon.mp4"
MODEL_PATH = "footballdetm2.pt"
OUTPUT_PATH = "output_custom_14.mp4"

CONF_THRESHOLD = 0.3
BALL_CONF_THRESHOLD = 0.25
DUPLICATE_IOU_THRESHOLD = 0.4
ROLE_HISTORY_LEN = 15          # necha kadr tarixi asosida rol qarori qabul qilinadi

# --- Goalkeeper zona qoidasi ---
ENABLE_GOALKEEPER_FIX = True
# Ekran kengligining necha foizi "darvoza zonasi" hisoblanadi.
# Haqiqiy darvozabon "player"ga aylanib qolsa - bu qiymatni oshiring (masalan 0.18-0.20)
GOALKEEPER_ZONE_WIDTH_RATIO = 0.15

# --- Past ishonchdagilar uchun rang orqali hakam aniqlash ---
ENABLE_REFEREE_FIX = True
REFEREE_COLOR_DIST_THRESHOLD = 70
ROLE_FIX_CONF_THRESHOLD = 0.5   # shu qiymatdan past ishonchdagilarga rang heuristikasi qo'llaniladi

ROLE_LETTER = {"player": "P", "goalkeeper": "G", "referee": "R", "ball": "B", "other": "O"}
TEAM_COLORS = [(0, 0, 255), (255, 100, 0)]        # qizil, ko'k (BGR)
ROLE_COLORS = {
    "goalkeeper": (0, 255, 0),     # yashil
    "referee": (0, 165, 255),      # to'q sariq
    "other": (128, 128, 128),      # kulrang
}
BALL_COLOR = (0, 255, 255)          # sariq


# ---------------------------------------------------------
# 1) DUBLIKAT BOX FILTRI
# ---------------------------------------------------------
def filter_overlapping_boxes(boxes_list, iou_threshold=0.4):
    """Bir xil klassdagi, ustma-ust tushgan box'larning IKKALASINI HAM olib tashlaydi.
    boxes_list elementlari: (cls_id, conf, track_id, x1, y1, x2, y2)"""
    def iou(b1, b2):
        x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
        x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union = area1 + area2 - inter
        return inter / union if union > 0 else 0

    n = len(boxes_list)
    to_remove = set()
    for i in range(n):
        for j in range(i + 1, n):
            if boxes_list[i][0] != boxes_list[j][0]:
                continue
            box_i = boxes_list[i][3:7]
            box_j = boxes_list[j][3:7]
            if iou(box_i, box_j) > iou_threshold:
                to_remove.add(i)
                to_remove.add(j)

    return [box for idx, box in enumerate(boxes_list) if idx not in to_remove]


# ---------------------------------------------------------
# 3) ROL SILLIQLASH (vaqt bo'yicha)
# ---------------------------------------------------------
class RoleSmoother:
    """Har bir track_id uchun oxirgi N kadrdagi rol tarixini saqlaydi va
    "ko'pchilik ovozi" bilan barqaror rolni qaytaradi."""
    def __init__(self, history_len=15):
        self.history = defaultdict(lambda: deque(maxlen=history_len))

    def update_and_get(self, track_id, current_role):
        if track_id == -1:
            return current_role  # tracking ID yo'q bo'lsa, silliqlab bo'lmaydi
        self.history[track_id].append(current_role)
        most_common_role, _ = Counter(self.history[track_id]).most_common(1)[0]
        return most_common_role


# ---------------------------------------------------------
# YORDAMCHI FUNKSIYALAR
# ---------------------------------------------------------
def get_shirt_color(frame, box):
    x1, y1, x2, y2 = map(int, box)
    h = y2 - y1
    shirt_y2 = max(y1 + int(h * 0.4), y1 + 1)
    crop = frame[y1:shirt_y2, x1:x2]
    if crop.size == 0:
        return np.array([128, 128, 128])
    return crop.reshape(-1, 3).mean(axis=0)


def draw_ellipse(frame, box, color, thickness=2):
    x1, y1, x2, y2 = map(int, box)
    center = (int((x1 + x2) / 2), y2)
    width = int(x2 - x1)
    cv2.ellipse(frame, center, (width // 2, int(width * 0.2)), 0, -45, 235, color, thickness, cv2.LINE_4)
    return frame


def draw_label(frame, box, text, bg_color):
    x1, y1, x2, y2 = map(int, box)
    brightness = 0.299 * bg_color[2] + 0.587 * bg_color[1] + 0.114 * bg_color[0]
    text_color = (0, 0, 0) if brightness > 180 else (255, 255, 255)
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cx = int((x1 + x2) / 2)
    top = max(y1 - th - 8, 0)
    cv2.rectangle(frame, (cx - tw // 2 - 4, top), (cx + tw // 2 + 4, top + th + 6), bg_color, -1)
    cv2.putText(frame, text, (cx - tw // 2, top + th + 1), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2, cv2.LINE_AA)
    return frame


# ---------------------------------------------------------
# 2) GOALKEEPER ZONA QOIDASI + PAST ISHONCHDAGILAR UCHUN RANG HEURISTIKASI
# ---------------------------------------------------------
def resolve_roles_for_frame(detections, frame, frame_width, team_centers):
    """
    detections - ro'yxat, har biri dict: {"role", "xyxy", "conf", "track_id"}
    (ball bu yerga kirmaydi, alohida ishlanadi)

    BOSQICH A - goalkeeper zona qoidasi (ISHONCHDAN QAT'I NAZAR):
        Har qanday "goalkeeper" deb topilgan detektsiya, agar darvoza
        zonasidan tashqarida bo'lsa - "player"ga aylantiriladi.

    BOSQICH B - past ishonchdagilar uchun rang+pozitsiya yordami:
        Faqat model ISHONCHI PAST (< ROLE_FIX_CONF_THRESHOLD) bo'lgan
        detektsiyalar uchun, rang farqi orqali hakam/darvozabon nomzodini
        aniqlaymiz.
    """
    if team_centers is None:
        return detections

    zone = frame_width * GOALKEEPER_ZONE_WIDTH_RATIO

    # --- BOSQICH A ---
    if ENABLE_GOALKEEPER_FIX:
        for d in detections:
            if d["role"] != "goalkeeper":
                continue
            cx = (d["xyxy"][0] + d["xyxy"][2]) / 2
            in_zone = cx < zone or cx > frame_width - zone
            if not in_zone:
                d["role"] = "player"

    # --- BOSQICH B ---
    if ENABLE_REFEREE_FIX:
        low_conf_dets = [d for d in detections if d["conf"] < ROLE_FIX_CONF_THRESHOLD]

        for d in low_conf_dets:
            shirt_color = get_shirt_color(frame, d["xyxy"])
            d["color_dist"] = np.linalg.norm(team_centers - shirt_color, axis=1).min()

        distinct = [d for d in low_conf_dets if d["color_dist"] > REFEREE_COLOR_DIST_THRESHOLD]
        distinct_ids = {id(d) for d in distinct}

        for d in low_conf_dets:
            d["role"] = "referee" if id(d) in distinct_ids else "player"

        if ENABLE_GOALKEEPER_FIX and distinct:
            left_candidates = [d for d in distinct if (d["xyxy"][0] + d["xyxy"][2]) / 2 < zone]
            right_candidates = [d for d in distinct if (d["xyxy"][0] + d["xyxy"][2]) / 2 > frame_width - zone]

            if left_candidates:
                gk = min(left_candidates, key=lambda d: (d["xyxy"][0] + d["xyxy"][2]) / 2)
                gk["role"] = "goalkeeper"
            if right_candidates:
                gk = max(right_candidates, key=lambda d: (d["xyxy"][0] + d["xyxy"][2]) / 2)
                gk["role"] = "goalkeeper"

    return detections


# ---------------------------------------------------------
# ASOSIY JARAYON
# ---------------------------------------------------------
def main():
    model = YOLO(MODEL_PATH)
    names = model.names
    role_smoother = RoleSmoother(history_len=ROLE_HISTORY_LEN)

    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    team_centers = None
    color_samples = []

    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    results_gen = model.track(
        source=VIDEO_PATH, tracker="bytetrack.yaml", conf=CONF_THRESHOLD, iou=0.5, stream=True,
    )

    frame_idx = 0
    for result in results_gen:
        frame = result.orig_img.copy()
        boxes = result.boxes

        # --- xom aniqlashlarni yig'amiz ---
        raw_list = []
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                raw_list.append((cls_id, conf, track_id, x1, y1, x2, y2))

        # --- 1) DUBLIKAT FILTRI ---
        filtered_list = filter_overlapping_boxes(raw_list, DUPLICATE_IOU_THRESHOLD)

        # --- dastlabki kadrlarda jamoa ranglarini yig'amiz ---
        if team_centers is None:
            for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
                if names[cls_id] == "player":
                    color_samples.append(get_shirt_color(frame, (x1, y1, x2, y2)))
            if frame_idx > 30 and len(color_samples) > 10:
                samples = np.array(color_samples, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
                _, _, centers = cv2.kmeans(samples, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
                team_centers = centers

        # --- ball va boshqalarni ajratamiz ---
        best_ball = None
        detections = []
        for cls_id, conf, track_id, x1, y1, x2, y2 in filtered_list:
            raw_role = names[cls_id]
            if raw_role == "ball":
                if conf >= BALL_CONF_THRESHOLD and (best_ball is None or conf > best_ball[1]):
                    best_ball = (raw_role, conf, track_id, x1, y1, x2, y2)
                continue
            detections.append({
                "role": raw_role, "xyxy": np.array([x1, y1, x2, y2]),
                "conf": conf, "track_id": track_id,
            })

        # --- 2) ZONA QOIDASI + RANG HEURISTIKASI (bir kadr ichida) ---
        detections = resolve_roles_for_frame(detections, frame, w, team_centers)

        # --- 3) ROL SILLIQLASH (vaqt bo'yicha, barcha tuzatishlardan keyin) ---
        for d in detections:
            d["role"] = role_smoother.update_and_get(d["track_id"], d["role"])

        # --- chizamiz ---
        for d in detections:
            role = d["role"]
            xyxy = d["xyxy"]
            conf = d["conf"]
            track_id = d["track_id"]

            team_idx = 0
            if role == "player" and team_centers is not None:
                shirt_color = get_shirt_color(frame, xyxy)
                dists = np.linalg.norm(team_centers - shirt_color, axis=1)
                team_idx = int(np.argmin(dists))

            color = TEAM_COLORS[team_idx] if role == "player" else ROLE_COLORS.get(role, (128, 128, 128))
            draw_ellipse(frame, xyxy, color)
            draw_label(frame, xyxy, f"{track_id}-{ROLE_LETTER.get(role, 'O')}-{int(conf * 100)}%", color)

        # --- to'p (soddalashtirilgan: to'rtburchak, ID yo'q, iz yo'q) ---
        if best_ball is not None:
            raw_role, conf, track_id, x1, y1, x2, y2 = best_ball
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), BALL_COLOR, 2)
            draw_label(frame, (x1, y1, x2, y2), f"{ROLE_LETTER['ball']}-{int(conf * 100)}%", BALL_COLOR)

        writer.write(frame)
        frame_idx += 1

    writer.release()
    print(f"[OK] Tayyor: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
import os
from IPython.display import Video

# Colab uchun mos formatga o'giramiz (brauzerda ko'rinishi uchun)
os.system('ffmpeg -y -i output_custom_14.mp4 -vcodec libx264 output_custom_14_converted.mp4')
Video("output_custom_14_converted.mp4", embed=True, width=800)